# Ensemble Forecasting & Uncertainty Quantification

Generate ensemble forecasts with Earth2Studio to visualize forecast uncertainty: spaghetti plots, probability maps, and spread-skill analysis.

**Extra install:** `uv add earth2studio --extra perturbation --extra statistics`

In [ ]:
import os
from datetime import datetime

import torch
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import earth2studio.run as run
from earth2studio.models.px import FCN
from earth2studio.data import GFS
from earth2studio.io import ZarrBackend
from earth2studio.perturbation import SphericalGaussian

In [ ]:
CONFIG = {
    "forecast_date": "2026-01-08",
    "nsteps": 10,
    "n_ensemble": 16,
    "output_root": "outputs/ensemble",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

os.makedirs(CONFIG["output_root"], exist_ok=True)
print(f"Device: {CONFIG['device']}")

## Run Ensemble Forecast

In [ ]:
fcn_model = FCN.load_model(FCN.load_default_package())
gfs_data = GFS()
perturbation = SphericalGaussian(noise_amplitude=0.05)

io_ens = ZarrBackend(
    f"{CONFIG['output_root']}/fcn_ensemble.zarr",
    backend_kwargs={"overwrite": True},
)

io_ens = run.ensemble(
    [CONFIG["forecast_date"]],
    CONFIG["nsteps"],
    CONFIG["n_ensemble"],
    fcn_model,
    gfs_data,
    io_ens,
    perturbation_method=perturbation,
)
ds = xr.open_zarr(f"{CONFIG['output_root']}/fcn_ensemble.zarr")
print(f"Ensemble forecast complete. Shape: {ds['t2m'].shape}")
print(f"Dimensions: {ds['t2m'].dims}")

## Spaghetti Plot

T2M evolution at a single location across all ensemble members.

In [ ]:
# Extract T2M at a specific location (New York City)
nyc_lat, nyc_lon = 40.7, -74.0
# Convert negative lon to 0-360 if needed
nyc_lon_360 = nyc_lon % 360

lats = ds["lat"].values
lons = ds["lon"].values
lat_idx = np.argmin(np.abs(lats - nyc_lat))
lon_idx = np.argmin(np.abs(lons - nyc_lon_360))

# Shape: (ensemble, time, lead_time, lat, lon) or similar
# Extract all members at this grid point
t2m_point = ds["t2m"].isel(lat=lat_idx, lon=lon_idx, time=0).values - 273.15
hours = np.arange(t2m_point.shape[-1]) * 6  # lead time in hours

print(f"Grid point: lat={lats[lat_idx]:.2f}, lon={lons[lon_idx]:.2f}")
print(f"Ensemble shape at point: {t2m_point.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Plot each member as a thin line
if t2m_point.ndim == 2:
    # (ensemble, lead_time)
    n_members = t2m_point.shape[0]
    for i in range(n_members):
        ax.plot(hours, t2m_point[i], color="steelblue", alpha=0.3, linewidth=0.8)
    ens_mean = t2m_point.mean(axis=0)
    ens_std = t2m_point.std(axis=0)
elif t2m_point.ndim == 1:
    # Single ensemble dimension may be flattened — adjust indexing
    n_members = CONFIG["n_ensemble"]
    lead_times = len(hours)
    t2m_reshaped = t2m_point.reshape(n_members, lead_times)
    for i in range(n_members):
        ax.plot(hours, t2m_reshaped[i], color="steelblue", alpha=0.3, linewidth=0.8)
    ens_mean = t2m_reshaped.mean(axis=0)
    ens_std = t2m_reshaped.std(axis=0)

ax.plot(hours, ens_mean, color="navy", linewidth=2.5, label="Ensemble Mean")
ax.fill_between(hours, ens_mean - ens_std, ens_mean + ens_std,
                color="steelblue", alpha=0.2, label="±1σ Spread")

ax.set_xlabel("Lead Time (hours)")
ax.set_ylabel("2m Temperature (°C)")
ax.set_title(f"Ensemble T2M Spaghetti Plot — NYC ({CONFIG['forecast_date']})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_root']}/spaghetti_nyc.png", dpi=150, bbox_inches="tight")
plt.show()

## Freezing Probability Map

Fraction of ensemble members predicting T2M below 0°C at +24h.

In [ ]:
step = 4  # +24h

# Get all ensemble members at this lead time
t2m_all = ds["t2m"].isel(time=0, lead_time=step).values - 273.15  # (ensemble, lat, lon)

# Probability of freezing = fraction of members below 0
if t2m_all.ndim == 3:
    prob_freeze = (t2m_all < 0).mean(axis=0)
else:
    prob_freeze = (t2m_all < 0).astype(float)

fig, ax = plt.subplots(figsize=(14, 7), subplot_kw={"projection": ccrs.Robinson()})
cf = ax.pcolormesh(
    lons, lats, prob_freeze,
    transform=ccrs.PlateCarree(), cmap="Blues", vmin=0, vmax=1, shading="auto",
)
cbar = plt.colorbar(cf, ax=ax, orientation="horizontal", pad=0.04, shrink=0.6)
cbar.set_label("P(T2M < 0°C)")
ax.coastlines()
ax.gridlines(linewidth=0.3, alpha=0.4)
ax.set_title(f"Freezing Probability — +{step*6}h  |  {CONFIG['n_ensemble']} members  |  Init: {CONFIG['forecast_date']}")
plt.savefig(f"{CONFIG['output_root']}/freeze_probability.png", dpi=150, bbox_inches="tight")
plt.show()

## Ensemble Spread Map

Standard deviation across members shows where the forecast is most uncertain.

In [ ]:
t2m_spread = ds["t2m"].isel(time=0, lead_time=step).values - 273.15
if t2m_spread.ndim == 3:
    spread = t2m_spread.std(axis=0)
else:
    spread = np.zeros_like(lats)  # fallback

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7),
    subplot_kw={"projection": ccrs.Robinson()})

# Panel 1: Ensemble mean
if t2m_all.ndim == 3:
    ens_mean_map = t2m_all.mean(axis=0)
else:
    ens_mean_map = t2m_all
im1 = ax1.pcolormesh(lons, lats, ens_mean_map, transform=ccrs.PlateCarree(),
                      cmap="RdBu_r", vmin=-40, vmax=40, shading="auto")
ax1.coastlines()
ax1.gridlines(linewidth=0.3, alpha=0.4)
ax1.set_title("Ensemble Mean T2M (°C)")
plt.colorbar(im1, ax=ax1, orientation="horizontal", pad=0.04, shrink=0.8)

# Panel 2: Spread
im2 = ax2.pcolormesh(lons, lats, spread, transform=ccrs.PlateCarree(),
                      cmap="YlOrRd", vmin=0, vmax=5, shading="auto")
ax2.coastlines()
ax2.gridlines(linewidth=0.3, alpha=0.4)
ax2.set_title("Ensemble Spread — Std Dev (°C)")
plt.colorbar(im2, ax=ax2, orientation="horizontal", pad=0.04, shrink=0.8)

fig.suptitle(f"Ensemble T2M — +{step*6}h  |  {CONFIG['n_ensemble']} members", fontsize=14, y=1.0)
plt.savefig(f"{CONFIG['output_root']}/mean_vs_spread.png", dpi=150, bbox_inches="tight")
plt.show()

## Spread Growth Over Lead Time

In [ ]:
spreads = []
for s in range(CONFIG["nsteps"] + 1):
    t2m_s = ds["t2m"].isel(time=0, lead_time=s).values - 273.15
    if t2m_s.ndim == 3:
        spreads.append(t2m_s.std(axis=0).mean())  # global mean spread
    else:
        spreads.append(0)

hours_all = np.arange(len(spreads)) * 6

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours_all, spreads, marker="o", color="darkorange", linewidth=2)
ax.set_xlabel("Lead Time (hours)")
ax.set_ylabel("Global Mean Ensemble Spread (°C)")
ax.set_title(f"T2M Ensemble Spread Growth — {CONFIG['n_ensemble']} members")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_root']}/spread_growth.png", dpi=150, bbox_inches="tight")
plt.show()